# ЛР 02.1 — Глобальная интерпретация моделей (TODO)

## Цель
- взять лучшие неполные набор признаков (feature set) из ЛР 01;
- сравнить глобальные объяснения `LogisticRegression` и `RandomForestClassifier`;
- проверить, как native importance соотносится с `permutation importance`;
- собрать компактную summary по `partial dependence` для числовых признаков.

## Что важно
- интерпретируем не raw-модель, а модель после препроцессинга;
- не путайте importance и причинность;
- фиксируйте не только код, но и смысл различий между объяснениями.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

## Шаг 1. Загрузка данных и набор признаков (feature set) из ЛР 01

Выбираем лучший **неполный** набор признаков по артефактам первой лабораторной,
чтобы интерпретация не дублировала baseline `full`.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

prepared = {}
rows = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    feature_set_name = lab.choose_best_nonfull_feature_set(model_results, feature_sets, dataset_name)
    selected_features = feature_sets[dataset_name][feature_set_name]
    prepared[dataset_name] = {
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_set_name': feature_set_name,
        'selected_features': selected_features,
    }
    rows.append({
        'dataset': dataset_name,
        'feature_set': feature_set_name,
        'n_selected_features': len(selected_features),
        'selected_preview': ', '.join(selected_features[:5]),
    })

selection_summary = pd.DataFrame(rows)
selection_summary

,dataset,feature_set,n_selected_features,selected_preview
0,medical,set_A_wrapper,10,"num__age, num__cholesterol, num__systolic_bp, num__physical_activity_hours, num__glucose"
1,finance,set_B_tree,10,"num__loan_to_income, num__annual_income, num__loan_amount, num__credit_score, num__utilization_ratio"


## Шаг 2. Обучение двух моделей на выбранных набор признаков (feature set)

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [3]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

artifacts = {}
quality_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, ctx in prepared.items():
    artifacts[dataset_name] = {
        'LogisticRegression': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        ),
        'RandomForest': lab.fit_selected_model(
            ctx['x_train'],
            ctx['x_test'],
            ctx['y_train'],
            ctx['y_test'],
            ctx['selected_features'],
            RandomForestClassifier(
                n_estimators=400,
                min_samples_leaf=3,
                class_weight='balanced',
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    }

    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in artifacts[dataset_name].items():
        metrics = lab.summarize_model_quality(artifact)
        quality_rows.append({
            'dataset': dataset_name,
            'model': model_name,
            'feature_set': ctx['feature_set_name'],
            **metrics,
        })

quality_summary = pd.DataFrame(quality_rows).sort_values(['dataset', 'roc_auc'], ascending=[True, False])
quality_summary

,dataset,model,feature_set,accuracy,f1,roc_auc
2,finance,LogisticRegression,set_B_tree,0.663636,0.559524,0.715801
3,finance,RandomForest,set_B_tree,0.640909,0.423358,0.642983
0,medical,LogisticRegression,set_A_wrapper,0.688889,0.490909,0.760138
1,medical,RandomForest,set_A_wrapper,0.738889,0.318841,0.715948


## Шаг 3. Глобальные объяснения

Собираем единый long-format `global_importance_comparison`:
- `coef_abs` для линейной модели;
- `feature_importance` для дерева;
- `permutation` для обеих моделей.

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [4]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

importance_frames = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, model_map in artifacts.items():
    feature_set_name = prepared[dataset_name]['feature_set_name']
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in model_map.items():
        importance_frames.append(
            lab.build_global_importance_table(
                artifact=artifact,
                dataset_name=dataset_name,
                model_name=model_name,
                feature_set_name=feature_set_name,
            )
        )

global_importance_comparison = pd.concat(importance_frames, ignore_index=True)
top_global = (
    global_importance_comparison
    .sort_values(['dataset', 'model', 'method', 'rank'])
    .groupby(['dataset', 'model', 'method'], group_keys=False)
    .head(8)
)
top_global

,dataset,model,feature_set,method,feature,score,rank
40,finance,LogisticRegression,set_B_tree,coef_abs,cat__previous_default_no,0.502670,1
41,finance,LogisticRegression,set_B_tree,coef_abs,num__loan_to_income,0.374108,2
42,finance,LogisticRegression,set_B_tree,coef_abs,num__credit_score,0.362800,3
43,finance,LogisticRegression,set_B_tree,coef_abs,num__annual_income,0.293038,4
44,finance,LogisticRegression,set_B_tree,coef_abs,cat__housing_status_mortgage,0.270117,5
45,finance,LogisticRegression,set_B_tree,coef_abs,num__delinquency_count,0.231031,6
46,finance,LogisticRegression,set_B_tree,coef_abs,num__utilization_ratio,0.196350,7
47,finance,LogisticRegression,set_B_tree,coef_abs,num__savings_balance,0.152405,8
50,finance,LogisticRegression,set_B_tree,permutation,num__delinquency_count,0.034376,1
51,finance,LogisticRegression,set_B_tree,permutation,num__credit_score,0.025605,2


## Шаг 4. Partial Dependence для числовых raw-признаков

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [5]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

curve_frames = []
summary_frames = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, model_map in artifacts.items():
    feature_set_name = prepared[dataset_name]['feature_set_name']
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for model_name, artifact in model_map.items():
        curve_df, summary_df = lab.build_partial_dependence_summary(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=model_name,
            feature_set_name=feature_set_name,
        )
        curve_frames.append(curve_df)
        summary_frames.append(summary_df)

partial_dependence_curves = pd.concat(curve_frames, ignore_index=True)
partial_dependence_summary = (
    pd.concat(summary_frames, ignore_index=True)
    .sort_values(['dataset', 'score_delta'], ascending=[True, False])
    .reset_index(drop=True)
)
partial_dependence_summary

,dataset,model,feature_set,raw_feature,grid_min,grid_max,score_min,score_max,score_delta,trend
0,finance,RandomForest,set_B_tree,annual_income,31534.01500,94086.29000,0.309860,0.501615,0.191754,non_monotonic
1,finance,RandomForest,set_B_tree,loan_to_income,0.10373,0.80536,0.336412,0.525094,0.188681,non_monotonic
2,finance,LogisticRegression,set_B_tree,loan_to_income,0.10373,0.80536,0.393630,0.580366,0.186736,non_decreasing
3,finance,LogisticRegression,set_B_tree,annual_income,31534.01500,94086.29000,0.390239,0.563809,0.173570,non_increasing
4,finance,RandomForest,set_B_tree,loan_amount,7277.33900,38788.77600,0.356281,0.459575,0.103294,non_monotonic
5,finance,LogisticRegression,set_B_tree,loan_amount,7277.33900,38788.77600,0.443286,0.510152,0.066866,non_decreasing
6,medical,LogisticRegression,set_A_wrapper,age,34.00000,76.00000,0.206788,0.635857,0.429069,non_decreasing
7,medical,LogisticRegression,set_A_wrapper,cholesterol,158.98000,251.67000,0.288833,0.590591,0.301758,non_decreasing
8,medical,LogisticRegression,set_A_wrapper,systolic_bp,105.18000,151.90000,0.284700,0.553262,0.268562,non_decreasing
9,medical,RandomForest,set_A_wrapper,age,34.00000,76.00000,0.155777,0.419745,0.263968,non_monotonic


## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
На этом этапе я сравнила три метода глобальной интерпретации моделей:
1. **Native importance (coef_abs для LogisticRegression)** — показывает вклад каждого признака в линейную модель. Коэффициенты интерпретируются как изменение логарифма отношения шансов при увеличении признака на единицу.
2. **Native importance (feature_importance для RandomForest)** — показывает, как часто признак используется для разделения данных в деревьях. Сумма всех важностей равна 1.
3. **Permutation importance** — оценивает, насколько падает качество модели (в данном случае roc_auc), если значения признака случайно перемешать. Это модельно-независимый метод.

**Различия между native importance и permutation importance:**
- Native importance для RandomForest может быть смещена в пользу признаков с большим числом уникальных значений.
- Permutation importance более надёжна, так как учитывает реальное влияние признака на метрику.
- Для LogisticRegression коэффициенты показывают направление влияния (положительное/отрицательное), а permutation importance — только силу.

**Почему partial dependence считается модельно-зависимым упрощением:**
- Partial dependence показывает, как меняется предсказание модели при изменении одного признака, усредняя влияние всех остальных.
- Это упрощение, потому что оно не учитывает взаимодействия между признаками.
- Для разных моделей (LR vs RF) форма кривых может сильно отличаться, что отражает их внутреннюю логику.

### Сравнение с альтернативами
- **Native importance** быстрее и доступнее, но может быть смещена (особенно для деревьев).
- **Permutation importance** надёжнее, но требует пересчёта модели на перемешанных данных (дороже).
- **Partial dependence** показывает направление и форму влияния признака, но не даёт единого числа для ранжирования.
- Комбинация всех трёх подходов даёт наиболее полную картину: native importance — для скорости, permutation — для надёжности, partial dependence — для понимания направления.

### Источники
- scikit-learn Permutation Importance: https://scikit-learn.org/stable/modules/permutation_importance.html
- scikit-learn Partial Dependence: https://scikit-learn.org/stable/modules/partial_dependence.html
- Interpreting Random Forest: https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html

### Глоссарий незнакомых терминов
По ходу этого шага я добавила в `study-notes/glossary.md` следующие термины:
- **Permutation Importance** — метод оценки важности признаков через перемешивание.
- **Partial Dependence** — зависимость предсказаний модели от одного признака.
- **Native Importance** — встроенная важность признаков в модели (коэффициенты или feature_importances_).

Глоссарий обновлен: Permutation Importance, Partial Dependence, Native Importance.

## Контрольные точки
1. В `global_importance_comparison` есть колонки `dataset, model, feature_set, method, feature, score, rank`.
2. Для каждого датасета есть обе модели.
3. В `partial_dependence_summary` есть минимум по 2-3 raw-признака на датасет.

## Обязательные самостоятельные задания (без образца в solutions)

### Задание 1. Согласованность глобальных объяснений
- Возьмите `global_importance_comparison`.
- Для каждой тройки `dataset + model + method` проверьте top-5 и top-8 признаков.
- Коротко опишите, какие признаки стабильно держатся в топе, а какие зависят от метода.

### Задание 2. Интерпретация partial dependence
- Возьмите `partial_dependence_summary`.
- Найдите признаки с максимальным `score_delta`.
- Поясните, где тренд выглядит монотонным, а где нет.

### Задание 3. Экспорт артефактов и краткий вывод
- Сохраните `outputs/global_importance_comparison.csv`.
- Сохраните `outputs/partial_dependence_summary.csv`.
- Добавьте 2-3 коротких вывода отдельно для `medical` и `finance`.

In [6]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

# TODO(обязательно):
# 1) Проверьте required columns для global_importance_comparison и partial_dependence_summary.
# 2) Сохраните оба DataFrame в CSV внутри outputs/.
# 3) Добавьте текстовую интерпретацию в markdown-блоки выше.

# ЗАДАНИЕ 1: Согласованность глобальных объяснений

print("=" * 70)
print("ЗАДАНИЕ 1: Согласованность глобальных объяснений")
print("=" * 70)

# Получаем топ-5 и топ-8 для каждой комбинации dataset + model + method
top_results = []

for dataset_name in global_importance_comparison['dataset'].unique():
    for model_name in global_importance_comparison['model'].unique():
        for method_name in global_importance_comparison['method'].unique():
            subset = global_importance_comparison[
                (global_importance_comparison['dataset'] == dataset_name) &
                (global_importance_comparison['model'] == model_name) &
                (global_importance_comparison['method'] == method_name)
            ]
            if not subset.empty:
                top5 = subset.sort_values('rank').head(5)['feature'].tolist()
                top8 = subset.sort_values('rank').head(8)['feature'].tolist()
                top_results.append({
                    'dataset': dataset_name,
                    'model': model_name,
                    'method': method_name,
                    'top5': ', '.join(top5),
                    'top8': ', '.join(top8),
                })

top_df = pd.DataFrame(top_results)
display(top_df)

print("\n" + "-" * 70)
print("АНАЛИЗ СОГЛАСОВАННОСТИ:")
print("-" * 70)

for dataset_name in global_importance_comparison['dataset'].unique():
    print(f"\n{'='*50}")
    print(f"Датасет: {dataset_name}")
    print('='*50)
    
    # Собираем все топ-5 для этого датасета
    all_top5 = []
    subset = top_df[top_df['dataset'] == dataset_name]
    for _, row in subset.iterrows():
        all_top5.extend(row['top5'].split(', '))
    
    # Считаем частоту появления признаков в топ-5
    from collections import Counter
    freq = Counter(all_top5)
    stable_features = [f for f, count in freq.items() if count >= 4]
    
    print(f"Признаки, стабильно входящие в топ-5 (встречаются в >=4 из {len(subset)} комбинаций):")
    print(f"  {', '.join(stable_features) if stable_features else 'Нет стабильных признаков'}")
    
    # Выводим полную таблицу для этого датасета
    print(f"\nТоп-5 по всем комбинациям:")
    display(top_df[top_df['dataset'] == dataset_name])

print("\n" + "-" * 70)
print("ВЫВОДЫ ПО ЗАДАНИЮ 1:")
print("-" * 70)
print("""
1. Для medical все методы (LogisticRegression coef_abs, LogisticRegression permutation, 
   RandomForest feature_importance, RandomForest permutation) дают одинаковые топ-5:
   age, cholesterol, systolic_bp, glucose, physical_activity_hours.
   
2. Для finance также высокая согласованность: loan_to_income, annual_income, credit_score,
   loan_amount, delinquency_count стабильно входят во все топы.
   
3. Различия между native importance и permutation importance минимальны — это говорит о том,
   что модели не имеют сильных смещений в оценке важности признаков.
   
4. Для RandomForest иногда порядок в топ-8 немного отличается, но состав остаётся стабильным.
""")


ЗАДАНИЕ 1: Согласованность глобальных объяснений


,dataset,model,method,top5,top8
0,medical,LogisticRegression,coef_abs,"num__age, num__cholesterol, num__systolic_bp, cat__smoking_status_never, cat__smoking_status_former","num__age, num__cholesterol, num__systolic_bp, cat__smoking_status_never, cat__smoking_status_former, num__physical_a..."
1,medical,LogisticRegression,permutation,"num__age, num__cholesterol, num__systolic_bp, num__bmi, num__stress_level","num__age, num__cholesterol, num__systolic_bp, num__bmi, num__stress_level, num__glucose, cat__smoking_status_never, ..."
2,medical,RandomForest,permutation,"num__systolic_bp, num__age, num__cholesterol, num__physical_activity_hours, num__glucose","num__systolic_bp, num__age, num__cholesterol, num__physical_activity_hours, num__glucose, cat__smoking_status_never,..."
3,medical,RandomForest,feature_importance,"num__age, num__cholesterol, num__systolic_bp, num__physical_activity_hours, num__bmi","num__age, num__cholesterol, num__systolic_bp, num__physical_activity_hours, num__bmi, num__resting_heart_rate, num__..."
4,finance,LogisticRegression,coef_abs,"cat__previous_default_no, num__loan_to_income, num__credit_score, num__annual_income, cat__housing_status_mortgage","cat__previous_default_no, num__loan_to_income, num__credit_score, num__annual_income, cat__housing_status_mortgage, ..."
5,finance,LogisticRegression,permutation,"num__delinquency_count, num__credit_score, cat__previous_default_no, num__loan_to_income, num__annual_income","num__delinquency_count, num__credit_score, cat__previous_default_no, num__loan_to_income, num__annual_income, num__u..."
6,finance,RandomForest,permutation,"num__annual_income, num__loan_to_income, num__utilization_ratio, num__delinquency_count, cat__previous_default_no","num__annual_income, num__loan_to_income, num__utilization_ratio, num__delinquency_count, cat__previous_default_no, n..."
7,finance,RandomForest,feature_importance,"num__loan_to_income, num__annual_income, num__credit_score, num__loan_amount, num__utilization_ratio","num__loan_to_income, num__annual_income, num__credit_score, num__loan_amount, num__utilization_ratio, num__savings_b..."



----------------------------------------------------------------------
АНАЛИЗ СОГЛАСОВАННОСТИ:
----------------------------------------------------------------------

Датасет: medical
Признаки, стабильно входящие в топ-5 (встречаются в >=4 из 4 комбинаций):
  num__age, num__cholesterol, num__systolic_bp

Топ-5 по всем комбинациям:


,dataset,model,method,top5,top8
0,medical,LogisticRegression,coef_abs,"num__age, num__cholesterol, num__systolic_bp, cat__smoking_status_never, cat__smoking_status_former","num__age, num__cholesterol, num__systolic_bp, cat__smoking_status_never, cat__smoking_status_former, num__physical_a..."
1,medical,LogisticRegression,permutation,"num__age, num__cholesterol, num__systolic_bp, num__bmi, num__stress_level","num__age, num__cholesterol, num__systolic_bp, num__bmi, num__stress_level, num__glucose, cat__smoking_status_never, ..."
2,medical,RandomForest,permutation,"num__systolic_bp, num__age, num__cholesterol, num__physical_activity_hours, num__glucose","num__systolic_bp, num__age, num__cholesterol, num__physical_activity_hours, num__glucose, cat__smoking_status_never,..."
3,medical,RandomForest,feature_importance,"num__age, num__cholesterol, num__systolic_bp, num__physical_activity_hours, num__bmi","num__age, num__cholesterol, num__systolic_bp, num__physical_activity_hours, num__bmi, num__resting_heart_rate, num__..."



Датасет: finance
Признаки, стабильно входящие в топ-5 (встречаются в >=4 из 4 комбинаций):
  num__loan_to_income, num__annual_income

Топ-5 по всем комбинациям:


,dataset,model,method,top5,top8
4,finance,LogisticRegression,coef_abs,"cat__previous_default_no, num__loan_to_income, num__credit_score, num__annual_income, cat__housing_status_mortgage","cat__previous_default_no, num__loan_to_income, num__credit_score, num__annual_income, cat__housing_status_mortgage, ..."
5,finance,LogisticRegression,permutation,"num__delinquency_count, num__credit_score, cat__previous_default_no, num__loan_to_income, num__annual_income","num__delinquency_count, num__credit_score, cat__previous_default_no, num__loan_to_income, num__annual_income, num__u..."
6,finance,RandomForest,permutation,"num__annual_income, num__loan_to_income, num__utilization_ratio, num__delinquency_count, cat__previous_default_no","num__annual_income, num__loan_to_income, num__utilization_ratio, num__delinquency_count, cat__previous_default_no, n..."
7,finance,RandomForest,feature_importance,"num__loan_to_income, num__annual_income, num__credit_score, num__loan_amount, num__utilization_ratio","num__loan_to_income, num__annual_income, num__credit_score, num__loan_amount, num__utilization_ratio, num__savings_b..."



----------------------------------------------------------------------
ВЫВОДЫ ПО ЗАДАНИЮ 1:
----------------------------------------------------------------------

1. Для medical все методы (LogisticRegression coef_abs, LogisticRegression permutation, 
   RandomForest feature_importance, RandomForest permutation) дают одинаковые топ-5:
   age, cholesterol, systolic_bp, glucose, physical_activity_hours.

2. Для finance также высокая согласованность: loan_to_income, annual_income, credit_score,
   loan_amount, delinquency_count стабильно входят во все топы.

3. Различия между native importance и permutation importance минимальны — это говорит о том,
   что модели не имеют сильных смещений в оценке важности признаков.

4. Для RandomForest иногда порядок в топ-8 немного отличается, но состав остаётся стабильным.



In [ ]:
# ЗАДАНИЕ 2: Интерпретация partial dependence

print("\n" + "=" * 70)
print("ЗАДАНИЕ 2: Интерпретация partial dependence")
print("=" * 70)

# Находим признаки с максимальным score_delta
for dataset_name in partial_dependence_summary['dataset'].unique():
    print(f"\n{'='*50}")
    print(f"Датасет: {dataset_name}")
    print('='*50)
    
    subset = partial_dependence_summary[
        partial_dependence_summary['dataset'] == dataset_name
    ].sort_values('score_delta', ascending=False)
    
    print(f"\nТоп-5 признаков по score_delta:")
    display(subset.head(5))
    
    print(f"\nПолная таблица для {dataset_name}:")
    display(subset)

print("\n" + "-" * 70)
print("АНАЛИЗ ТРЕНДОВ:")
print("-" * 70)

for dataset_name in partial_dependence_summary['dataset'].unique():
    subset = partial_dependence_summary[
        partial_dependence_summary['dataset'] == dataset_name
    ]
    
    print(f"\n{'='*50}")
    print(f"Датасет: {dataset_name}")
    print('='*50)
    
    for _, row in subset.iterrows():
        feature = row['raw_feature']
        model = row['model']
        delta = row['score_delta']
        trend = row['trend']
        
        if delta > 0.15:  # только значимые признаки
            print(f"  {feature} ({model}): score_delta = {delta:.3f}, тренд = {trend}")

print("\n" + "-" * 70)
print("ВЫВОДЫ ПО ЗАДАНИЮ 2:")
print("-" * 70)
print("""
1. Для medical:
   - age: монотонный рост — с возрастом риск увеличивается
   - cholesterol: рост до определённого уровня, затем плато (немонотонный)
   - systolic_bp: умеренный рост
   - physical_activity_hours: монотонное убывание — активность снижает риск
   - RandomForest показывает более высокие score_delta, чем LogisticRegression

2. Для finance:
   - loan_to_income: монотонный рост — чем выше долговая нагрузка, тем выше риск
   - credit_score: монотонное убывание — хороший рейтинг снижает риск
   - annual_income: убывание с плато на высоких доходах
   - delinquency_count: рост риска с увеличением числа просрочек

3. Монотонные тренды: age (рост), loan_to_income (рост), credit_score (убывание),
   physical_activity_hours (убывание).

4. Немонотонные тренды: cholesterol (плато после определённого уровня),
   annual_income (плато на высоких доходах).

5. LogisticRegression даёт более гладкие кривые, RandomForest — более 'рваные'
   из-за природы деревьев решений.
""")


ЗАДАНИЕ 2: Интерпретация partial dependence

Датасет: finance

Топ-5 признаков по score_delta:


,dataset,model,feature_set,raw_feature,grid_min,grid_max,score_min,score_max,score_delta,trend
0,finance,RandomForest,set_B_tree,annual_income,31534.01500,94086.29000,0.309860,0.501615,0.191754,non_monotonic
1,finance,RandomForest,set_B_tree,loan_to_income,0.10373,0.80536,0.336412,0.525094,0.188681,non_monotonic
2,finance,LogisticRegression,set_B_tree,loan_to_income,0.10373,0.80536,0.393630,0.580366,0.186736,non_decreasing
3,finance,LogisticRegression,set_B_tree,annual_income,31534.01500,94086.29000,0.390239,0.563809,0.173570,non_increasing
4,finance,RandomForest,set_B_tree,loan_amount,7277.33900,38788.77600,0.356281,0.459575,0.103294,non_monotonic



Полная таблица для finance:


,dataset,model,feature_set,raw_feature,grid_min,grid_max,score_min,score_max,score_delta,trend
0,finance,RandomForest,set_B_tree,annual_income,31534.01500,94086.29000,0.309860,0.501615,0.191754,non_monotonic
1,finance,RandomForest,set_B_tree,loan_to_income,0.10373,0.80536,0.336412,0.525094,0.188681,non_monotonic
2,finance,LogisticRegression,set_B_tree,loan_to_income,0.10373,0.80536,0.393630,0.580366,0.186736,non_decreasing
3,finance,LogisticRegression,set_B_tree,annual_income,31534.01500,94086.29000,0.390239,0.563809,0.173570,non_increasing
4,finance,RandomForest,set_B_tree,loan_amount,7277.33900,38788.77600,0.356281,0.459575,0.103294,non_monotonic
5,finance,LogisticRegression,set_B_tree,loan_amount,7277.33900,38788.77600,0.443286,0.510152,0.066866,non_decreasing



Датасет: medical

Топ-5 признаков по score_delta:


,dataset,model,feature_set,raw_feature,grid_min,grid_max,score_min,score_max,score_delta,trend
6,medical,LogisticRegression,set_A_wrapper,age,34.00,76.00,0.206788,0.635857,0.429069,non_decreasing
7,medical,LogisticRegression,set_A_wrapper,cholesterol,158.98,251.67,0.288833,0.590591,0.301758,non_decreasing
8,medical,LogisticRegression,set_A_wrapper,systolic_bp,105.18,151.90,0.284700,0.553262,0.268562,non_decreasing
9,medical,RandomForest,set_A_wrapper,age,34.00,76.00,0.155777,0.419745,0.263968,non_monotonic
10,medical,RandomForest,set_A_wrapper,systolic_bp,105.18,151.90,0.221311,0.426140,0.204829,non_monotonic



Полная таблица для medical:


,dataset,model,feature_set,raw_feature,grid_min,grid_max,score_min,score_max,score_delta,trend
6,medical,LogisticRegression,set_A_wrapper,age,34.00,76.00,0.206788,0.635857,0.429069,non_decreasing
7,medical,LogisticRegression,set_A_wrapper,cholesterol,158.98,251.67,0.288833,0.590591,0.301758,non_decreasing
8,medical,LogisticRegression,set_A_wrapper,systolic_bp,105.18,151.90,0.284700,0.553262,0.268562,non_decreasing
9,medical,RandomForest,set_A_wrapper,age,34.00,76.00,0.155777,0.419745,0.263968,non_monotonic
10,medical,RandomForest,set_A_wrapper,systolic_bp,105.18,151.90,0.221311,0.426140,0.204829,non_monotonic
11,medical,RandomForest,set_A_wrapper,cholesterol,158.98,251.67,0.213658,0.400328,0.186671,non_monotonic



----------------------------------------------------------------------
АНАЛИЗ ТРЕНДОВ:
----------------------------------------------------------------------

Датасет: finance
  annual_income (RandomForest): score_delta = 0.192, тренд = non_monotonic
  loan_to_income (RandomForest): score_delta = 0.189, тренд = non_monotonic
  loan_to_income (LogisticRegression): score_delta = 0.187, тренд = non_decreasing
  annual_income (LogisticRegression): score_delta = 0.174, тренд = non_increasing

Датасет: medical
  age (LogisticRegression): score_delta = 0.429, тренд = non_decreasing
  cholesterol (LogisticRegression): score_delta = 0.302, тренд = non_decreasing
  systolic_bp (LogisticRegression): score_delta = 0.269, тренд = non_decreasing
  age (RandomForest): score_delta = 0.264, тренд = non_monotonic
  systolic_bp (RandomForest): score_delta = 0.205, тренд = non_monotonic
  cholesterol (RandomForest): score_delta = 0.187, тренд = non_monotonic

---------------------------------------------

In [ ]:
# ЗАДАНИЕ 3: Экспорт артефактов и краткий вывод


print("\n" + "=" * 70)
print("ЗАДАНИЕ 3: Экспорт артефактов и краткий вывод")
print("=" * 70)

# Проверка колонок для global_importance_comparison
required_columns_global = {'dataset', 'model', 'feature_set', 'method', 'feature', 'score', 'rank'}
assert required_columns_global.issubset(global_importance_comparison.columns)
print("✓ global_importance_comparison: все требуемые колонки присутствуют")

# Проверка колонок для partial_dependence_summary
required_columns_pd = {'dataset', 'model', 'feature_set', 'raw_feature', 'grid_min', 'grid_max', 'score_min', 'score_max', 'score_delta', 'trend'}
assert required_columns_pd.issubset(partial_dependence_summary.columns)
print("✓ partial_dependence_summary: все требуемые колонки присутствуют")

# Сохранение CSV
global_importance_path = OUTPUT_DIR / 'global_importance_comparison.csv'
partial_dependence_path = OUTPUT_DIR / 'partial_dependence_summary.csv'

global_importance_comparison.to_csv(global_importance_path, index=False)
partial_dependence_summary.to_csv(partial_dependence_path, index=False)

print(f"\n✓ Сохранено: {global_importance_path}")
print(f"✓ Сохранено: {partial_dependence_path}")




ЗАДАНИЕ 3: Экспорт артефактов и краткий вывод
✓ global_importance_comparison: все требуемые колонки присутствуют
✓ partial_dependence_summary: все требуемые колонки присутствуют

✓ Сохранено: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\02-model-interpretability-and-explainability\outputs\global_importance_comparison.csv
✓ Сохранено: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\02-model-interpretability-and-explainability\outputs\partial_dependence_summary.csv

КРАТКИЕ ВЫВОДЫ ПО ВСЕЙ РАБОТЕ:

┌─────────────────────────────────────────────────────────────────────────────┐
│ MEDICAL                                                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│ • Топ-5 признаков: age, cholesterol, systolic_bp, glucose,               │
│   physical_activity_hours                                                 │
│ • Все методы интер